# 분류 모델 평가 실습

이 파일은 분류 모델의 평가 지표를 직접 확인하는 실습용 파일이다.

실습 구성
1. 이진 분류 평가 - 유방암 진단
2. 다중 분류 평가 - 아이리스

실습 목표
1. 분류 문제에서 train/test 분리 후 모델을 학습할 수 있다.
2. confusion matrix를 통해 어떤 예측이 맞고 틀렸는지 해석할 수 있다.
3. accuracy, precision, recall, f1-score의 의미를 결과와 함께 해석할 수 있다.
4. 이진 분류에서는 ROC Curve와 AUC를 확인할 수 있다.
5. 다중 분류에서는 classification report와 macro 평균의 의미를 확인할 수 있다.

진행 가이드
1. 각 문제의 요구사항을 먼저 읽는다.
2. 코드 셀은 직접 작성한다.
3. 출력 결과를 보고 해석 질문까지 답해본다.
4. 필요하면 셀을 추가해도 된다.


In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_curve,
    roc_auc_score
)

plt.rc('font', family='Malgun Gothic')
plt.rc('axes', unicode_minus=False)


## 1. 이진 분류 평가 - 유방암 진단

실습 목적
- 악성(malignant) / 양성(benign) 두 클래스를 구분하는 이진 분류 문제를 다룬다.
- 정확도만 보는 것이 아니라 정밀도, 재현율, F1-score, ROC-AUC까지 함께 확인한다.
- 특히 의료 진단과 같이 놓치면 안 되는 문제에서 재현율이 왜 중요한지 생각해본다.

요구사항
1. `load_breast_cancer()`로 데이터를 불러온다.
2. 입력 데이터와 타깃 데이터의 shape를 확인한다.
3. DataFrame으로 변환하여 앞부분을 확인한다.
4. `train_test_split()`으로 학습/테스트 데이터를 분리한다.
   - `test_size=0.2`
   - `random_state=42`
   - `stratify=target`
5. `LogisticRegression` 모델을 학습한다.
6. 테스트 데이터 예측값과 양성 확률값을 구한다.
7. 아래 평가를 수행한다.
   - confusion matrix
   - accuracy
   - precision
   - recall
   - f1-score
   - classification report
8. ROC Curve를 그리고 AUC를 계산한다.

해석 질문
1. confusion matrix에서 FP와 FN은 각각 무엇을 의미하는가?
2. 의료 진단 문제에서 precision과 recall 중 어떤 값을 더 중요하게 볼 수 있는가?
3. AUC 값이 높다는 것은 무엇을 의미하는가?

In [3]:
# 1) 데이터 로드
from sklearn.datasets import load_breast_cancer
import pandas as pd

df = load_breast_cancer()
X = df.data
y = df.target
print(X.shape)
print(y.shape)

(569, 30)
(569,)


In [4]:
# 2) 데이터프레임으로 확인
df = pd.DataFrame(X, columns=df.feature_names)
df['target'] = y
df.head()


,mean radius,mean texture,mean perimeter,mean area,mean smoothness,mean compactness,mean concavity,mean concave points,mean symmetry,mean fractal dimension,...,worst texture,worst perimeter,worst area,worst smoothness,worst compactness,worst concavity,worst concave points,worst symmetry,worst fractal dimension,target
0,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,0.2419,0.07871,...,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890,0
1,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,0.1812,0.05667,...,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902,0
2,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,0.2069,0.05999,...,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758,0
3,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,0.2597,0.09744,...,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300,0
4,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,0.1809,0.05883,...,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678,0


In [5]:
# 3) train / test 분리
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)
print(X_train.shape, y_train.shape)
print(X_test.shape, y_test.shape)


(455, 30) (455,)
(114, 30) (114,)


In [7]:
# 4) 모델 학습
from sklearn.linear_model import LogisticRegression

lr_clf = LogisticRegression(max_iter=3000)
lr_clf.fit(X_train,y_train)

print(lr_clf.score(X_train, y_train))
print(lr_clf.score(X_test, y_test))

0.9560439560439561
0.9649122807017544


In [8]:
# 5) 예측값 / 예측확률 구하기
print(lr_clf.predict(X_test))

print(lr_clf.predict_proba(X_test))

[0 1 0 1 0 1 1 0 0 0 1 0 1 0 0 1 0 1 1 1 0 0 1 1 1 1 0 1 1 1 1 1 1 1 0 1 1
 1 1 0 1 1 1 0 0 1 1 1 1 0 1 1 1 1 1 1 1 0 0 1 1 1 1 1 0 1 1 1 1 1 1 1 1 0
 0 0 0 1 1 1 1 1 0 1 0 1 1 1 1 1 1 1 0 0 0 1 0 1 0 1 0 0 0 1 0 0 1 0 1 0 1
 0 1 1]
[[1.00000000e+00 3.41854046e-11]
 [3.43057332e-05 9.99965694e-01]
 [9.49729909e-01 5.02700914e-02]
 [3.96603627e-01 6.03396373e-01]
 [9.99999998e-01 2.13672683e-09]
 [1.73094243e-02 9.82690576e-01]
 [2.89760692e-05 9.99971024e-01]
 [9.99990129e-01 9.87055830e-06]
 [9.99950268e-01 4.97322663e-05]
 [1.00000000e+00 1.43015915e-10]
 [1.35707559e-03 9.98642924e-01]
 [9.95116087e-01 4.88391344e-03]
 [8.87561440e-04 9.99112439e-01]
 [9.99988668e-01 1.13324788e-05]
 [9.99428691e-01 5.71309097e-04]
 [6.09161995e-02 9.39083800e-01]
 [7.80430875e-01 2.19569125e-01]
 [2.13778186e-02 9.78622181e-01]
 [1.42178512e-03 9.98578215e-01]
 [2.04233975e-02 9.79576602e-01]
 [9.96198764e-01 3.80123605e-03]
 [9.87564066e-01 1.24359344e-02]
 [3.33062472e-04 9.99666938e-01]
 [7.4120

In [10]:
# 6) confusion matrix 확인
from sklearn.metrics import confusion_matrix
print(confusion_matrix(y_test, lr_clf.predict(X_test)))

print((39 + 71) / (39 + 3 + 1 + 71))

[[39  3]
 [ 1 71]]
0.9649122807017544


In [11]:
# 7) 평가지표 계산
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

y_pred = lr_clf.predict(X_test)
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1-score:", f1_score(y_test, y_pred))

Accuracy: 0.9649122807017544
Precision: 0.9594594594594594
Recall: 0.9861111111111112
F1-score: 0.9726027397260274


In [14]:
# 8) classification report 출력
from sklearn.metrics import classification_report
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.97      0.93      0.95        42
           1       0.96      0.99      0.97        72

    accuracy                           0.96       114
   macro avg       0.97      0.96      0.96       114
weighted avg       0.97      0.96      0.96       114



In [ ]:
# 9) ROC Curve 시각화 및 AUC 계산


## 2. 다중 분류 평가 - 아이리스

실습 목적
- setosa, versicolor, virginica 세 품종을 구분하는 다중 분류 문제를 다룬다.
- 다중 분류에서는 클래스가 여러 개이므로 confusion matrix를 표 형태로 해석하는 연습이 중요하다.
- classification report에서 클래스별 precision, recall, f1-score를 비교해본다.

요구사항
1. `load_iris()`로 데이터를 불러온다.
2. 입력 데이터와 타깃 데이터의 shape를 확인한다.
3. `train_test_split()`으로 학습/테스트 데이터를 분리한다.
   - `test_size=0.2`
   - `random_state=42`
   - `stratify=y`
4. `LogisticRegression` 모델을 학습한다.
5. 테스트 데이터 예측값을 구한다.
6. confusion matrix를 출력하고 DataFrame 형태로 보기 좋게 정리한다.
7. classification report를 출력한다.
8. accuracy, precision, recall, f1-score를 `average='macro'`로 계산한다.

해석 질문
1. 어떤 클래스에서 오분류가 발생했는가?
2. macro average를 사용하는 이유는 무엇인가?
3. 이진 분류와 달리 다중 분류 평가에서 주의할 점은 무엇인가?


In [ ]:
# 1) 데이터 로드


In [ ]:
# 2) train / test 분리


In [ ]:
# 3) 모델 학습


In [ ]:
# 4) 예측값 구하기


In [ ]:
# 5) confusion matrix 출력


In [ ]:
# 6) confusion matrix를 DataFrame으로 정리


In [ ]:
# 7) classification report 출력


In [ ]:
# 8) macro 평균 기준 평가지표 계산


## 체크

1. 이진 분류에서 `predict()`와 `predict_proba()`를 구분해서 사용했는가?
2. ROC Curve의 x축이 FPR, y축이 TPR임을 확인했는가?
3. 다중 분류에서 precision, recall, f1-score 계산 시 `average` 옵션을 올바르게 사용했는가?
4. 결과 숫자만 출력하지 말고, 어떤 의미인지 해석까지 해보았는가?